# 🎬 CineVerify-Core-3B-v5 Training Notebook
### Evidence-Grounded Indian Cinema Temporal Verification SLM

- **Base Architecture:** `unsloth/Qwen2.5-3B-Instruct`
- **Dataset:** `cineverify_train_v5.jsonl` (train) + `cineverify_val_v5.jsonl` (disjoint holdout validation)
- **Key V5 Improvements:** Evidence-grounded stances (`[1]`, `[2]`), entity/sequel preservation, chronological temporal progression (`Day 10 ₹900 Cr vs Day 12 ₹1,200 Cr -> TRUE`), metric disambiguation, and all 6 verdict classes (`TRUE`, `FALSE`, `MISLEADING`, `OUTDATED`, `PARTIALLY_TRUE`, `UNVERIFIED`).
- **Note:** Uses conservative learning rate (`1.5e-4`, 2 epochs) + LoRA dropout `0.05` + validation eval loss checks to prevent catastrophic overfitting.

In [ ]:
!nvidia-smi
!pip install -U --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U unsloth_zoo
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets sentencepiece protobuf

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1536
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
import os, json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in examples["messages"]]
    return {"text": texts}

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

assert os.path.exists("cineverify_train_v5.jsonl"), "Upload cineverify_train_v5.jsonl and cineverify_val_v5.jsonl first!"
train_ds = Dataset.from_list(load_jsonl("cineverify_train_v5.jsonl")).map(formatting_prompts_func, batched=True)
val_ds = Dataset.from_list(load_jsonl("cineverify_val_v5.jsonl")).map(formatting_prompts_func, batched=True)
print(f"Loaded {len(train_ds)} train samples and {len(val_ds)} disjoint validation samples.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=1.5e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        save_strategy="steps",
        save_steps=25,
        load_best_model_at_end=True,
        optim="adamw_8bit",
        weight_decay=0.02,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs_v5",
        report_to="none",
    ),
)
trainer_stats = trainer.train()
print("V5 Training finished:", trainer_stats)

In [ ]:
# Export V5 Quantized GGUF without overwriting V4 until evaluated
model.save_pretrained_gguf("CineVerify-v5-GGUF", tokenizer, quantization_method="q4_k_m")
print("Exported CineVerify-v5-GGUF! Register locally via: ollama create cineverify-v5 -f Modelfile_v5")